# Day 25: Designing an SDK & API Platform

**Week 4 — System Design**

---

## 📚 Theory: API Design & Developer Experience

If you are developing the **Google Agent Development Kit (ADK)**, you aren't just building backend infrastructure; you are building tools for *other developers*. SDK (Software Development Kit) and API design requires a very specific set of system design principles centered around backward compatibility, security, and developer experience (DX).

### Principles of Great API Design
1. **RESTful Naming**: Use nouns, not verbs. (`POST /agents`, not `POST /createAgent`).
2. **Idempotency**: Retrying a failed request should be safe. If a developer calls `DELETE /agents/123` twice due to a network blip, the system state should be the same, and no unexpected errors should blow up the client.
3. **Pagination**: Never return a raw array of thousands of items. Use cursor-based pagination (`?cursor=xyz&limit=50`) for scalable iteration.
4. **Rate Limiting Headers**: Always return `X-RateLimit-Limit`, `X-RateLimit-Remaining`, and `X-RateLimit-Reset` so developers can programmatically pause their SDKs instead of guessing.

### Principles of Great SDK Design
1. **Language Idioms**: A Python SDK should feel "Pythonic" (using kwargs, generators, context managers). A Java SDK should feel "Java-like" (Builders, strong typing). Don't just auto-generate one generic wrapper for all languages.
2. **Sensible Defaults**: "Make the easy things easy, and the hard things possible." A developer should be able to launch a basic agent in 3 lines of code.
3. **Built-in Retries**: The SDK should handle transient network errors (`503 Service Unavailable`, `429 Too Many Requests`) automatically using Exponential Backoff.


## 🔨 Exercise: Architecting an Agent ADK

Imagine you are tasked with designing the public API and SDK architecture for a new Google Agent Platform.

### 1. Handling Backward Compatibility
**The Problem**: You decide to change how agents handle "Memory". If you change the JSON schema for the `/agents/create` endpoint, you will break the code of thousands of companies relying on your V1 SDK.

**The Solution (API Versioning)**:
Always version APIs. You can do this via the URL (`/v1/agents`) or via Headers (`Accept: application/vnd.google.v1+json`). 

If you must make a breaking change:
1. Release `v2`.
2. Mark `v1` as **Deprecated** (but keep it fully operational!).
3. Monitor logs to see which customers are still using `v1`.
4. Establish a strict Sunset policy (e.g., "We will support v1 for 18 months").

### 2. Webhooks & Asynchronous SDKs
**The Problem**: As discussed in Day 24, Agent executions are slow. If an SDK calls `agent.execute(prompt="Write a book")`, that HTTP request will time out.

**The Solution**:
Provide three ways for developers to handle long-running tasks in your SDK:
1. **Polling**: The SDK method `agent.execute_and_wait()` automatically sends a request, gets a `task_id`, and runs a `while True: sleep(2); check_status()` loop under the hood.
2. **Webhooks**: The developer provides a callback URL (`POST /agents/execute {webhook_url: "https://my-app.com/callback"}`). When the Orchestrator finishes the task, it sends an HTTP POST with the results to the developer's server.
3. **Streaming / WebSockets**: The SDK establishes a Server-Sent Events (SSE) connection so the developer can stream the agent's thought process (e.g., "Tool invoked...", "Searching web...") to their frontend UI in real-time.

### 3. API Gateway Deep Dive for Public APIs

When exposing an ADK to the public internet, your API Gateway must be significantly more robust than an internal one.

```mermaid
graph TD
    Developer[Developer SDK] --> WAF[Web Application Firewall]
    WAF --> Gateway[API Gateway]
    Gateway --> Auth[(Auth Service / IAM)]
    Gateway --> Quota[(Quota & Billing Service)]
    Gateway --> RateLimit[(Redis Rate Limiter)]
    Gateway --> Service[Internal Agent Service]
```

**Authentication & Authorization**:
The SDK should allow initialization with a simple API Key or OAuth token. The API gateway verifies this token against the Auth service. 

**Billing & Quotas**:
LLM execution is expensive! The Gateway must intercept the request, query the Billing service (e.g., "Does this user have credits remaining?"), and reject it with a `402 Payment Required` if they are out of funds. It must also log the usage so the customer is charged appropriately based on tokens consumed.

## 📝 Day 25 Review

### Concepts Validated Today
- [ ] The principles of RESTful API design (Idempotency, Pagination).
- [ ] How to design an SDK that provides high-level abstractions (like built-in exponential backoff and polling).
- [ ] Strategies for API Versioning and Backward Compatibility.
- [ ] Managing long-running API tasks using Webhooks and Server-Sent Events (SSE).
- [ ] The role of an API Gateway in public-facing platforms (Auth, Billing, Rate Limiting).

### Tomorrow's Preview
**Day 26: Observability and Monitoring** — Once a system is built, how do you know if it's broken? We will cover Logging, Metrics, and Distributed Tracing.